# yeast5: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('43_yeast5_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,67.760,57.630,76.762,47.222,99.583,73.403,75.100,98.161,62.695
smote,92.809,65.110,51.952,88.611,97.361,92.986,68.605,98.845,78.960
bsmote,93.930,65.982,52.310,90.833,97.361,94.097,69.247,98.913,79.956
ros,94.380,64.216,49.834,91.944,97.049,94.497,69.543,98.836,79.298
adasyn,93.945,66.441,53.059,90.833,97.396,94.115,69.916,98.884,80.193
casb,93.322,64.708,51.023,89.722,97.257,93.490,70.365,98.862,79.015
casb_relaxed,92.231,64.317,51.264,87.500,97.326,92.413,70.320,98.857,78.274
casb_edited,94.365,63.555,49.012,91.944,96.944,94.444,69.620,98.899,78.960
casb_ros,93.254,63.577,49.632,89.722,97.118,93.420,71.187,98.844,78.415


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,74.894,63.810,75.333,56.944,99.375,78.160,77.871,98.072,69.352
smote,95.443,63.843,48.652,94.167,96.840,95.503,68.347,97.883,79.643
bsmote,95.956,63.833,48.376,95.278,96.736,96.007,66.928,97.856,79.895
ros,96.193,66.974,52.000,95.278,97.222,96.250,68.802,97.910,81.583
adasyn,96.466,63.386,47.488,96.389,96.632,96.510,68.346,97.859,79.926
casb,96.620,65.495,49.927,96.389,96.944,96.667,69.675,97.951,81.058
casb_relaxed,95.462,63.928,48.656,94.167,96.875,95.521,69.758,97.881,79.695
casb_edited,96.905,62.292,46.012,97.500,96.389,96.944,62.291,98.958,79.598
casb_ros,96.193,66.974,52.000,95.278,97.222,96.250,68.766,97.907,81.583


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,75.205,66.176,79.643,57.500,99.549,78.524,79.543,99.092,70.691
smote,92.342,77.634,71.305,86.389,98.854,92.622,81.385,99.245,84.988
bsmote,92.971,78.101,71.216,87.500,98.819,93.160,79.851,99.246,85.536
ros,90.411,75.256,70.128,82.917,98.819,90.868,77.878,99.115,82.833
adasyn,93.492,76.811,68.469,88.611,98.646,93.628,82.710,99.275,85.151
casb,91.100,76.691,71.266,84.028,98.889,91.458,80.421,99.241,83.896
casb_relaxed,92.427,78.652,72.752,86.389,98.958,92.674,81.764,99.261,85.540
casb_edited,94.027,76.505,67.279,89.722,98.576,94.149,75.594,99.126,85.266
casb_ros,91.718,77.711,72.494,85.139,98.924,92.031,80.534,99.211,84.715


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,64.245,51.350,70.667,43.056,99.375,71.215,71.938,98.814,57.797
smote,97.368,60.135,43.464,98.889,95.903,97.396,69.393,98.725,78.752
bsmote,95.877,57.946,41.936,96.250,95.729,95.990,68.827,98.701,76.911
ros,97.289,60.275,43.818,98.750,95.903,97.326,70.409,98.761,78.782
adasyn,95.438,57.806,41.798,95.278,95.799,95.538,69.523,98.701,76.622
casb,96.651,59.568,43.349,97.500,95.903,96.701,70.450,98.779,78.109
casb_relaxed,97.911,60.354,43.456,100.000,95.868,97.934,69.740,98.733,79.132
casb_edited,97.221,58.980,42.234,98.750,95.764,97.257,69.991,98.730,78.100
casb_ros,97.255,59.590,42.974,98.750,95.833,97.292,70.939,98.791,78.423


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,70.526,59.741,75.601,51.181,99.470,75.326,76.113,98.535,65.133
smote,94.491,66.680,53.843,92.014,97.240,94.627,71.933,98.674,80.586
bsmote,94.684,66.465,53.460,92.465,97.161,94.813,71.213,98.679,80.575
ros,94.568,66.680,53.945,92.222,97.248,94.735,71.658,98.655,80.624
adasyn,94.835,66.111,52.704,92.778,97.118,94.948,72.624,98.680,80.473
casb,94.423,66.615,53.891,91.910,97.248,94.579,72.728,98.708,80.519
casb_relaxed,94.508,66.813,54.032,92.014,97.257,94.635,72.896,98.683,80.660
casb_edited,95.629,65.333,51.134,94.479,96.918,95.699,69.374,98.928,80.481
casb_ros,94.605,66.963,54.275,92.222,97.274,94.748,72.856,98.688,80.784


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,94.365,ros,94.380,-0.016,LOSS
1,f1,casb,64.708,adasyn,66.441,-1.733,LOSS
2,precision,casb_relaxed,51.264,adasyn,53.059,-1.795,LOSS
3,recall,casb_edited,91.944,ros,91.944,0.000,TIE


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,96.905,adasyn,96.466,0.439,WIN
1,f1,casb_ros,66.974,ros,66.974,0.000,TIE
2,precision,casb_ros,52.000,ros,52.000,0.000,TIE
3,recall,casb_edited,97.500,adasyn,96.389,1.111,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,94.027,adasyn,93.492,0.534,WIN
1,f1,casb_relaxed,78.652,bsmote,78.101,0.551,WIN
2,precision,casb_relaxed,72.752,smote,71.305,1.447,WIN
3,recall,casb_edited,89.722,adasyn,88.611,1.111,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,95.629,adasyn,94.835,0.794,WIN
1,f1,casb_ros,66.963,smote,66.680,0.283,WIN
2,precision,casb_ros,54.275,ros,53.945,0.330,WIN
3,recall,casb_edited,94.479,adasyn,92.778,1.701,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.665508,0.571429,0.800000,0.444444,0.996528,0.720486,0.815300,0.990741,0.618468,None
1,42,1,svm,smote,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.766534,0.991898,0.830931,None
2,42,1,svm,bsmote,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.764153,0.991512,0.830931,None
3,42,1,svm,ros,0.50,0.932936,0.695652,0.571429,0.888889,0.979167,0.934028,0.761989,0.991127,0.814294,None
4,42,1,svm,adasyn,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.764153,0.991512,0.830931,None
5,42,1,svm,casb,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.772090,0.992670,0.830931,None
6,42,1,svm,casb_relaxed,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.769166,0.992284,0.830931,None
7,42,1,svm,casb_edited,0.50,0.932936,0.695652,0.571429,0.888889,0.979167,0.934028,0.797873,0.991898,0.814294,None
8,42,1,svm,casb_ros,0.50,0.934589,0.727273,0.615385,0.888889,0.982639,0.935764,0.769166,0.992284,0.830931,None
9,42,1,knn,original,0.50,0.665508,0.571429,0.800000,0.444444,0.996528,0.720486,0.698445,0.992670,0.618468,None
